# AI-Based Network Intrusion Detection Using K-Means and Naive Bayes

This notebook follows the analysis from dataset inspection to model comparison. It uses the project modules so results are reproducible and preprocessing is fitted only on training data. The metrics are from this local experiment, not the published paper.

Place `KDDTrain+.txt` and `KDDTest+.txt` in `data/` before running the cells.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA

project_root = Path.cwd()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data import CLASS_NAMES, FEATURE_NAMES, load_datasets
from src.evaluation import compute_metrics, plot_confusion_matrix
from src.model import (fit_kmeans, get_cluster_distribution,
                       train_baseline_nb_model, train_decision_tree_model,
                       train_nb_kmeans_model, transform_distance_features)
from src.preprocessing import NUMERICAL_FEATURES, Preprocessor

sns.set_theme(style='whitegrid')

## 1. Load the dataset

The loader validates the expected NSL-KDD columns and rejects attack labels it cannot map. It does not download or create dataset records.

In [ ]:
try:
    train_df, test_df = load_datasets(project_root / 'data')
except FileNotFoundError as error:
    raise FileNotFoundError(
        "Place KDDTrain+.txt and KDDTest+.txt in the project's data/ folder. See data/README.md."
    ) from error

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
print(f'Feature count: {len(FEATURE_NAMES)}')
print('Columns:', FEATURE_NAMES)
print('Training missing values:', int(train_df.isna().sum().sum()))
print('Training duplicate rows:', int(train_df.duplicated().sum()))

## 2. Exploratory data analysis

The loader maps individual attack names into Normal, DoS, Probe, R2L, and U2R. `httptunnel` is grouped as R2L, consistent with the standard NSL-KDD grouping.

In [ ]:
print('Raw attack labels:', train_df['attack_type'].nunique())
display(train_df[FEATURE_NAMES].describe().T)
display(train_df['label_5class'].value_counts().reindex(CLASS_NAMES, fill_value=0).rename('records'))
display(train_df['attack_type'].value_counts().head(20).rename('records'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
train_df['label_5class'].value_counts().reindex(CLASS_NAMES, fill_value=0).plot.bar(
    ax=axes[0], color='#267a70', title='Training records by class')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Records')
train_df['attack_type'].value_counts().head(15).sort_values().plot.barh(
    ax=axes[1], color='#d17a42', title='15 most frequent attack labels')
axes[1].set_xlabel('Records')
correlation_features = NUMERICAL_FEATURES[:12]
sns.heatmap(train_df[correlation_features].corr(), cmap='vlag', center=0, ax=axes[2])
axes[2].set_title('Correlation: first 12 numeric features')
plt.tight_layout()
plt.show()

## 3. Preprocess features

The scaler and one-hot encoder are fitted on the training partition only. The same fitted transformations are applied to the test partition; unseen categorical values are ignored by the encoder.

In [ ]:
preprocessor = Preprocessor()
X_train = preprocessor.fit_transform(train_df)
X_test = preprocessor.transform(test_df)
y_train = train_df['label_5class'].to_numpy()
y_test = test_df['label_5class'].to_numpy()
print('Preprocessed train:', X_train.shape)
print('Preprocessed test: ', X_test.shape)

## 4. Fit K-Means and create distance features

K-Means is trained only on the preprocessed training set. Its cluster IDs are arbitrary. Each record is represented by its Euclidean distance to each of the five fitted centroids.

In [ ]:
kmeans = fit_kmeans(X_train, n_clusters=5, random_state=42)
cluster_sizes = get_cluster_distribution(kmeans, X_train)
display(pd.Series({i: cluster_sizes.get(i, 0) for i in range(5)}, name='records').rename_axis('cluster_id'))
X_train_dist = transform_distance_features(kmeans, X_train)
X_test_dist = transform_distance_features(kmeans, X_test)
print('Distance features:', X_train_dist.shape, X_test_dist.shape)

## 5. Train and evaluate classifiers

The proposed classifier uses the five distances. The baseline Naive Bayes and Decision Tree use the preprocessed original features. Evaluation is performed on KDDTest+.

In [ ]:
models = {
    'K-Means + Naive Bayes (Proposed)': (train_nb_kmeans_model(X_train_dist, y_train), X_test_dist),
    'Naive Bayes Baseline': (train_baseline_nb_model(X_train, y_train), X_test),
    'Decision Tree Baseline': (train_decision_tree_model(X_train, y_train, random_state=42), X_test),
}
results = {}
for name, (model, features) in models.items():
    predictions = model.predict(features)
    results[name] = compute_metrics(y_test, predictions, model_name=name, classes=CLASS_NAMES)

summary = pd.DataFrame([
    {key: value for key, value in result.items() if key in
     ('model_name', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
      'f1_weighted', 'detection_rate', 'false_positive_rate')}
    for result in results.values()
])
display(summary.set_index('model_name'))
proposed = results['K-Means + Naive Bayes (Proposed)']
display(pd.DataFrame(proposed['per_class_report']).T)
print('Detection rate (attack recall):', proposed['detection_rate'])
print('False-positive rate (normal predicted as attack):', proposed['false_positive_rate'])
plot_confusion_matrix(proposed['confusion_matrix'], CLASS_NAMES,
                      title='K-Means + Naive Bayes: KDDTest+ confusion matrix')
plt.show()

## 6. PCA visualization and conclusion

PCA below is a two-dimensional visualization of the five distance features, not an additional model input. A fixed-size random sample keeps the plot responsive.

In [ ]:
sample_size = min(5000, len(X_train_dist))
sample_indices = train_df.sample(n=sample_size, random_state=42).index.to_numpy()
# Dataset indices are positional in the NSL-KDD files.
sample_positions = train_df.index.get_indexer(sample_indices)
distance_sample = X_train_dist[sample_positions]
pca_coordinates = PCA(n_components=2, random_state=42).fit_transform(distance_sample)
sample_clusters = kmeans.predict(X_train[sample_positions]).astype(str)
pca_frame = pd.DataFrame({'PC1': pca_coordinates[:, 0], 'PC2': pca_coordinates[:, 1],
                          'Cluster': sample_clusters})
sns.scatterplot(data=pca_frame, x='PC1', y='PC2', hue='Cluster',
                palette='tab10', alpha=0.65, s=18)
plt.title('PCA projection of centroid-distance features')
plt.tight_layout()
plt.show()

In [ ]:
print('Conclusion')
print('The comparison above reports this run on the supplied NSL-KDD files.')
print('These are not the published results and do not claim to reproduce the reference paper.')
print('Review the test metrics and class-level errors before drawing conclusions about the distance representation.')